# Паттерн 4: Handoffs — передача управления через Command

Handoffs — это паттерн мультиагентной системы, в котором каждый агент сам решает, кому передать управление. Нет центрального контроллера — агенты напрямую маршрутизируют запрос через `Command(goto="имя_агента")`. Triage-агент анализирует запрос и передаёт специалисту. Специалист техподдержки обнаруживает, что вопрос связан с оплатой, и передаёт коллеге из billing. Billing видит, что клиент описывает баг, и возвращает в техподдержку. Цепочка передач — динамическая, непредсказуемая заранее.

```mermaid
graph LR
    START((START)) --> triage(["🔹 Triage<br/><i>передаёт управление</i>"])
    triage -->|"Command(goto=...)"| tech_support(["🔹 Tech Support<br/><i>передаёт управление</i>"])
    triage -->|"Command(goto=...)"| billing(["🔹 Billing<br/><i>передаёт управление</i>"])
    tech_support -->|"Command(goto=...)"| billing
    tech_support -->|"Command(goto=...)"| END((END))
    billing -->|"Command(goto=...)"| tech_support
    billing -->|"Command(goto=...)"| END

    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class triage,tech_support,billing worker
    class START,END terminal
```

In [ ]:
import sys
sys.path.insert(0, "..")

import operator
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [ ]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

## Состояние и LLM

Состояние `HandoffState` содержит четыре поля. Поле `history` использует редьюсер `operator.add`: каждый агент добавляет запись о своём действии, и LangGraph автоматически конкатенирует списки вместо перезаписи. Поле `current_agent` отслеживает, кто сейчас обрабатывает запрос, а `resolution` хранит финальный ответ клиенту.

In [ ]:
llm = get_llm()

class HandoffState(TypedDict):
    query: str                                         # Запрос клиента
    history: Annotated[list[str], operator.add]        # Лог действий (аддитивный)
    current_agent: str                                 # Текущий обработчик
    resolution: str                                    # Итоговый ответ

## Triage-узел: классификация запроса

Triage-агент анализирует запрос клиента и передаёт его нужному специалисту. Ключевой момент — аннотация возвращаемого типа `Command[Literal["tech_support", "billing"]]`. Это не просто подсказка для IDE: LangGraph использует `Literal[...]` для построения графа, определяя допустимые цели передачи. Без этой аннотации граф не будет знать о возможных переходах.

`Command` позволяет не только указать цель (`goto`), но и обновить состояние (`update`). Агент передаёт не просто управление, а контекст — запись о своём решении добавляется в `history`.

In [ ]:
def triage_node(state: HandoffState) -> Command[Literal["tech_support", "billing"]]:
    """Triage: анализирует запрос и передаёт нужному специалисту."""
    response = llm.invoke([
        SystemMessage(content="""Ты — маршрутизатор службы поддержки.
Определи тип запроса клиента:
- Технический вопрос (ошибки, настройки, подключение) → ответь ОДНИМ словом: tech
- Вопрос по оплате, счетам, подписке → ответь ОДНИМ словом: billing"""),
        HumanMessage(content=state["query"])
    ])

    content = response.content.strip().lower()
    if "billing" in content:
        target = "billing"
    else:
        target = "tech_support"

    print(f"  [Triage] Запрос классифицирован → {target}")
    return Command(
        goto=target,
        update={
            "history": [f"[Triage] Классификация → {target}"],
            "current_agent": target,
        }
    )

## Техподдержка: решение или передача

Техподдержка решает технические вопросы, но может обнаружить, что проблема связана с оплатой. В этом случае она передаёт управление в billing через `Command(goto="billing")`. Если вопрос решён — агент завершает граф через `Command(goto=END)`, что эквивалентно `goto="__end__"`. Обратите внимание на аннотацию `Command[Literal["billing", "__end__"]]` — она перечисляет обе допустимые цели.

In [ ]:
def tech_support_node(state: HandoffState) -> Command[Literal["billing", "__end__"]]:
    """
    Техподдержка: решает технические вопросы.
    Может передать в billing, если вопрос связан с оплатой.
    """
    response = llm.invoke([
        SystemMessage(content="""Ты — специалист технической поддержки.

Если вопрос технический — дай краткий ответ (2-3 предложения).
Если вопрос связан с оплатой/подпиской — ответь ОДНИМ словом: HANDOFF_BILLING

Будь дружелюбен и конкретен."""),
        HumanMessage(content=f"Запрос клиента: {state['query']}")
    ])

    content = response.content
    if "HANDOFF_BILLING" in content:
        print("  [Техподдержка] Вопрос по оплате — передаю в billing")
        return Command(
            goto="billing",
            update={
                "history": [f"[Техподдержка] → передал в billing"],
                "current_agent": "billing",
            }
        )

    print(f"  [Техподдержка] Ответ: {content[:80]}...")
    return Command(
        goto=END,
        update={
            "resolution": content,
            "history": [f"[Техподдержка] Решено"],
        }
    )

## Billing: зеркальная логика передачи

Billing-агент зеркально повторяет логику техподдержки: решает вопросы по оплате, но может передать в tech_support, если обнаружит технический вопрос. Именно эта двусторонняя связь делает паттерн Handoffs по-настоящему децентрализованным — агенты образуют сеть, а не иерархию.

In [ ]:
def billing_node(state: HandoffState) -> Command[Literal["tech_support", "__end__"]]:
    """
    Billing: решает вопросы по оплате.
    Может передать в техподдержку, если вопрос технический.
    """
    response = llm.invoke([
        SystemMessage(content="""Ты — специалист по оплате и подпискам.

Если вопрос по оплате/счетам/подписке — дай краткий ответ (2-3 предложения).
Если вопрос технический — ответь ОДНИМ словом: HANDOFF_TECH

Будь дружелюбен и конкретен."""),
        HumanMessage(content=f"Запрос клиента: {state['query']}")
    ])

    content = response.content
    if "HANDOFF_TECH" in content:
        print("  [Billing] Технический вопрос — передаю в техподдержку")
        return Command(
            goto="tech_support",
            update={
                "history": [f"[Billing] → передал в tech_support"],
                "current_agent": "tech_support",
            }
        )

    print(f"  [Billing] Ответ: {content[:80]}...")
    return Command(
        goto=END,
        update={
            "resolution": content,
            "history": [f"[Billing] Решено"],
        }
    )

## Сборка графа: без рёбер между узлами

При использовании `Command` явные рёбра между рабочими узлами не нужны — маршрутизация происходит изнутри каждого узла. Единственное ребро — от `START` к `triage`. Всё остальное определяется аннотациями `Command[Literal[...]]` в возвращаемых типах функций. LangGraph извлекает допустимые переходы из этих аннотаций при компиляции графа.

In [ ]:
graph = StateGraph(HandoffState)

graph.add_node("triage", triage_node)
graph.add_node("tech_support", tech_support_node)
graph.add_node("billing", billing_node)

graph.add_edge(START, "triage")
# Рёбра не добавляем — Command(goto=...) делает всю маршрутизацию

app = graph.compile(checkpointer=MemorySaver())

## Запуск

Параметр `recursion_limit` здесь критичен. Без него два агента могут бесконечно передавать управление друг другу: техподдержка решает, что вопрос по оплате, billing решает, что вопрос технический — и так по кругу. Ограничение в 10 шагов служит обязательной страховкой от таких осцилляций.

Запрос намеренно выбран пограничным — «не могу подключиться к серверу, при этом вчера оплатил подписку» — чтобы продемонстрировать, как triage классифицирует неоднозначный вопрос и как специалисты могут передавать его друг другу.

In [ ]:
config = {
    "configurable": {"thread_id": "handoff-001"},
    "recursion_limit": 10,  # Страховка от осцилляций между агентами
}

result = app.invoke(
    {
        "query": "Не могу подключиться к серверу, при этом вчера оплатил подписку",
        "history": [],
        "current_agent": "",
        "resolution": "",
    },
    config=config,
)

print(f"\n  Маршрут: {' → '.join(result['history'])}")
print(f"  Ответ: {result.get('resolution', 'Нет ответа')[:200]}")

## Итог

Паттерн Handoffs через `Command(goto=...)` отличается от Supervisor тремя ключевыми свойствами:

1. **Децентрализация** — нет единого управляющего узла; каждый агент сам решает, кому передать управление.
2. **Граф без рёбер** — маршрутизация закодирована внутри узлов через `Command`, а не через явные рёбра графа. LangGraph извлекает структуру из аннотаций `Command[Literal[...]]`.
3. **Обновление состояния при передаче** — `Command(goto=..., update=...)` позволяет передать не только управление, но и контекст: агент добавляет свою диагностику в историю.

Главный риск — осцилляции: два агента могут бесконечно передавать запрос друг другу. Обязательно используйте `recursion_limit` как страховку.